# Test-7: English to Odia Transformer
## Comprehensive Model Parameters, Training Convergence & Empirical Evaluation
**Course / Project Specification (§5.6) Audit & Analysis**

This notebook provides an interactive walkthrough and empirical audit of the from-scratch Sequence-to-Sequence Transformer trained for **English to Odia (ଓଡ଼ିଆ)** Neural Machine Translation (NMT).

> **Tip on viewing graphs**: In Jupyter or VS Code, click **Run All** (or press `Shift + Enter` on the code cells) to generate all live charts. The pre-rendered loss curve is also embedded below.

### Contents:
1. **Environment Setup & Artifact Loading**
2. **Model Architecture & Layer-by-Layer Parameter Breakdown** (Verifying the 4,005,696 parameters)
3. **Dataset Splits & Odia Subword Tokenization EDA** (Samanantar corpus & Brahmic script asymmetry)
4. **Training History, Loss Curves & Perplexity** (Smooth convergence & causal mask bug verification)
5. **Evaluation Metrics & SacreBLEU Breakdown** (N-gram precisions, brevity penalty, ratio)
6. **Qualitative Translation Samples** (5 test samples, including the $\ge 90$th percentile long sentence)
7. **Length vs. Quality Degradation Analysis** (Empirical analysis of capacity bottlenecks)
8. **Cross-Attention Alignment Heatmap** (Visualizing source-target token attention)
9. **Final Summary, Technical Q&A & Recommendations**

In [ ]:
import sys
import json
import math
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ensure repository root is on sys.path
REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
import torch.nn as nn

# Clean plot styling
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "sans-serif"]
plt.rcParams["axes.edgecolor"] = "#CBD5E1"
plt.rcParams["axes.linewidth"] = 0.8

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Repository root: {REPO_ROOT}")

## 1. Model Architecture & Layer-by-Layer Parameter Breakdown

The project implements the Transformer architecture described in **Section 5.6** of the assignment:
- **Sinusoidal Positional Encoding** + **Token Embedding** scaled by $\sqrt{d_{\text{model}}}$
- **$N=2$ Encoder Blocks**: Multi-head self-attention ($h=4, d=128$) $\rightarrow$ Residual + Post-LayerNorm $\rightarrow$ FFN ($d_{\text{ff}}=512$) $\rightarrow$ Residual + Post-LayerNorm
- **$N=2$ Decoder Blocks**: Masked causal self-attention $\rightarrow$ Residual + LN $\rightarrow$ Cross-attention over encoder representations $\rightarrow$ Residual + LN $\rightarrow$ FFN $\rightarrow$ Residual + LN
- **Linear Output Projection**: Untied `Linear(128, 8000)` projection to Odia vocabulary logits

Let's verify the exact layer-by-layer parameter counts against the theoretical formulas and the PyTorch implementation.

In [ ]:
from configs.base import (
    D_MODEL, N_HEADS, D_FF, N_ENCODER_LAYERS, N_DECODER_LAYERS,
    DROPOUT, EN_VOCAB_SIZE, OR_VOCAB_SIZE, TIE_OUTPUT_PROJECTION
)
from src.model.transformer import Seq2SeqTransformer

# Instantiate model
model = Seq2SeqTransformer(
    src_vocab_size=EN_VOCAB_SIZE,
    tgt_vocab_size=OR_VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    d_ff=D_FF,
    n_encoder_layers=N_ENCODER_LAYERS,
    n_decoder_layers=N_DECODER_LAYERS,
    dropout=DROPOUT,
    tie_output_projection=TIE_OUTPUT_PROJECTION
)

# Tabulate parameters by module
param_details = []
for name, p in model.named_parameters():
    param_details.append({
        "Module": name,
        "Shape": list(p.shape),
        "Parameters": p.numel(),
        "Requires Grad": p.requires_grad
    })

df_params = pd.DataFrame(param_details)

# Group high-level modules
def categorize_module(name):
    if "encoder.embeddings" in name:
        return "Source Embeddings (en)"
    elif "encoder.layers" in name:
        return "Encoder Blocks (N=2)"
    elif "decoder.embeddings" in name:
        return "Target Embeddings (or)"
    elif "decoder.layers" in name:
        return "Decoder Blocks (N=2)"
    elif "output_projection" in name:
        return "Output Projection"
    return "Other"

df_params["Category"] = df_params["Module"].apply(categorize_module)
summary_table = df_params.groupby("Category")["Parameters"].sum().reset_index()
total_params = df_params["Parameters"].sum()

print(f"=== Total Trainable Model Parameters: {total_params:,} ===")
display(summary_table.style.format({"Parameters": "{:,}"}))

# Visualization of parameter distribution
fig, ax = plt.subplots(figsize=(8, 3.8), dpi=100)
colors = ["#0D9488", "#14B8A6", "#0284C7", "#38BDF8", "#F59E0B"]
bars = ax.barh(summary_table["Category"], summary_table["Parameters"], color=colors, edgecolor="#0F172A", alpha=0.9)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 25000, bar.get_y() + bar.get_height()/2, f"{w:,} ({w/total_params:.1%})", va="center", fontsize=9, fontweight="bold")
ax.set_xlim(0, max(summary_table["Parameters"]) * 1.3)
ax.set_title("Parameter Distribution by Component", fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Number of Parameters")
plt.tight_layout()
plt.show()

## 2. Dataset Splits & Odia Subword Tokenization EDA

The dataset is derived from `ai4bharat/samanantar` (English–Odia config `or`):
- **Candidate Pool**: 58,000 raw pairs streamed and normalized
- **Text Normalization**: Unicode NFC normalization, zero-width space/BOM removal, and conjunct preservation
- **Length Filtering**: Dropped pairs exceeding `MAX_LEN=64` subwords
- **Final Splits**: 36,000 Train / 2,000 Validation / 2,000 Test (zero sentence overlap)

### Odia vs. English Subword Disparity (Extra Credit Feature)
Odia uses a Brahmic script with rich inflectional morphology and multi-byte UTF-8 encoding. Even with an identical 8,000-subword vocabulary, Odia sentences require roughly **$3\times$ as many tokens** as English sentences.

In [ ]:
from project_data import DATA_STATS, TOKENIZER_STATS

# 1. Dataset Split Summary
splits_data = {
    "Split": ["Train", "Validation", "Test", "Total Filtered Survivors", "Initial Candidate Pool"],
    "Sentence Pairs": [DATA_STATS["train_size"], DATA_STATS["val_size"], DATA_STATS["test_size"], DATA_STATS["retention_survivors"], DATA_STATS["candidate_pool_size"]],
    "Percentage": [
        f"{DATA_STATS['train_size']/DATA_STATS['total_size']:.1%}",
        f"{DATA_STATS['val_size']/DATA_STATS['total_size']:.1%}",
        f"{DATA_STATS['test_size']/DATA_STATS['total_size']:.1%}",
        f"{DATA_STATS['retention_rate_pct']:.1f}% retention",
        "100%"
    ]
}
display(pd.DataFrame(splits_data))

# 2. Token Length Distribution: English vs. Odia
en_stats = TOKENIZER_STATS["english"]
or_stats = TOKENIZER_STATS["odia"]

token_stats_df = pd.DataFrame({
    "Metric": ["Mean", "Median", "90th Percentile (p90)", "95th Percentile (p95)", "99th Percentile (p99)", "Max Length"],
    "English Subwords": [en_stats["mean"], en_stats["median"], en_stats["p90"], en_stats["p95"], en_stats["p99"], en_stats["max"]],
    "Odia Subwords": [or_stats["mean"], or_stats["median"], or_stats["p90"], or_stats["p95"], or_stats["p99"], or_stats["max"]]
})
display(token_stats_df)

# 3. Retention Rate vs. MAX_LEN Threshold
retention = TOKENIZER_STATS["retention_at_max_len"]
max_lens = [int(k) for k in retention.keys()]
rates = [retention[k] for k in retention.keys()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2), dpi=100)

# Subplot 1: Retention Curve
ax1.plot(max_lens, rates, marker="o", color="#0D9488", linewidth=2, markersize=6)
ax1.axvline(64, color="#EF4444", linestyle="--", label=f"Chosen MAX_LEN=64 ({retention['64']}%) ")
ax1.set_title("Pair Retention Rate vs. MAX_LEN Filter", fontsize=11, fontweight="bold")
ax1.set_xlabel("Max Subword Tokens (including <SOS>/<EOS>)")
ax1.set_ylabel("Pair Retention (%)")
ax1.set_ylim(30, 105)
ax1.legend()

# Subplot 2: Subword Token Count Comparison
metrics_to_plot = ["Mean", "Median", "p90", "p95", "p99"]
x = np.arange(len(metrics_to_plot))
width = 0.35
en_vals = [en_stats["mean"], en_stats["median"], en_stats["p90"], en_stats["p95"], en_stats["p99"]]
or_vals = [or_stats["mean"], or_stats["median"], or_stats["p90"], or_stats["p95"], or_stats["p99"]]

ax2.bar(x - width/2, en_vals, width, label="English (en)", color="#3B82F6")
ax2.bar(x + width/2, or_vals, width, label="Odia (or)", color="#F97316")
ax2.set_title("Subword Token Count Comparison (8k Vocab)", fontsize=11, fontweight="bold")
ax2.set_xticks(x)
ax2.set_xticklabels(metrics_to_plot)
ax2.set_ylabel("Subwords per Sentence")
ax2.legend()

plt.tight_layout()
plt.show()

## 3. Training History, Loss Curves & Perplexity

The model was trained for **40 epochs** on Kaggle CPU with:
- **Optimizer**: Adam ($\beta_1=0.9, \beta_2=0.98, \epsilon=10^{-9}$)
- **Learning Rate Schedule**: Noam inverse-square-root warmup schedule (900 warmup steps)
- **Regularization**: Label smoothing ($0.1$) + Gradient clipping (norm 1.0) + Dropout ($0.1$)
- **Causal Masking Check**: As specified in the prompt (*"watch for the causal mask bug - if val loss is suspiciously perfect, your decoder is peeking!"*), the loss curve shows validation loss steadily tracking above training loss without any sudden collapse to near-zero.

### Pre-rendered Loss Curve:
![Training & Validation Loss Curve](reports/figures/loss_curve.png)

In [ ]:
from project_data import TRAINING_HISTORY

df_history = pd.DataFrame(TRAINING_HISTORY)
df_history["train_ppl"] = np.exp(df_history["train_loss"])
df_history["val_ppl"] = np.exp(df_history["val_loss"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.2), dpi=100)

# Loss curves
ax1.plot(df_history["epoch"], df_history["train_loss"], label="Train Loss", color="#0D9488", linewidth=2)
ax1.plot(df_history["epoch"], df_history["val_loss"], label="Validation Loss", color="#F59E0B", linewidth=2)
ax1.set_title("Cross-Entropy Loss (with 0.1 Label Smoothing)", fontsize=11, fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.6)

# Perplexity curves
ax2.plot(df_history["epoch"], df_history["train_ppl"], label="Train Perplexity", color="#0D9488", linewidth=2)
ax2.plot(df_history["epoch"], df_history["val_ppl"], label="Validation Perplexity", color="#F59E0B", linewidth=2)
ax2.set_title("Token-Level Perplexity (PPL = exp(loss))", fontsize=11, fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Perplexity")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

# Display milestones table
milestones = df_history[df_history["epoch"].isin([1, 5, 10, 20, 30, 40])].copy()
display(milestones.style.format({
    "train_loss": "{:.4f}",
    "val_loss": "{:.4f}",
    "train_ppl": "{:.2f}",
    "val_ppl": "{:.2f}"
}))

## 4. Evaluation Metrics & SacreBLEU Breakdown

Evaluation was conducted over the full **2,000-sentence test set** using `sacrebleu`.

### Corpus BLEU Score: **2.60**
- **BLEU Signature**: `BLEU = 2.60 22.2/4.9/1.7/0.5 (BP = 0.829, ratio = 0.842, hyp_len = 12904, ref_len = 15317)`
- **Comparison**:
  - Baseline (18 epochs, no label smoothing, greedy): **2.19 BLEU**
  - Enhanced (40 epochs, label smoothing 0.1, 3-gram repetition blocking): **2.60 BLEU** (+18.7% relative gain)

In [ ]:
from project_data import EVAL_RESULTS

print(f"Overall Test Corpus BLEU: {EVAL_RESULTS['bleu_score']:.2f}")
print(f"SacreBLEU Signature: {EVAL_RESULTS['bleu_signature']}")
print(f"Total Test Examples Evaluated: {EVAL_RESULTS['num_test_examples']:,}")
print(f"Evaluation Decoding Time: {EVAL_RESULTS['decode_seconds']:.1f}s")

# Precision breakdown
precisions = [22.2, 4.9, 1.7, 0.5]
ngrams = ["1-gram (Unigram)", "2-gram (Bigram)", "3-gram (Trigram)", "4-gram (4-gram)"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2), dpi=100)

# Precision bars
bars = ax1.bar(ngrams, precisions, color=["#0D9488", "#14B8A6", "#06B6D4", "#0EA5E9"], edgecolor="#0F172A", alpha=0.9)
for bar in bars:
    y = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, y + 0.6, f"{y}%", ha="center", fontweight="bold")
ax1.set_title("N-gram Precision Breakdown", fontsize=11, fontweight="bold")
ax1.set_ylabel("Precision (%)")
ax1.set_ylim(0, 27)

# Comparison: Baseline vs Enhanced
runs = ["Baseline (18 ep, no smoothing)", "Enhanced (40 ep + smoothing + rep-block)"]
bleu_vals = [2.19, 2.60]
bars2 = ax2.bar(runs, bleu_vals, color=["#94A3B8", "#0D9488"], edgecolor="#0F172A", width=0.5)
for bar in bars2:
    y = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, y + 0.08, f"BLEU {y:.2f}", ha="center", fontweight="bold")
ax2.set_title("Model Checkpoint Comparison", fontsize=11, fontweight="bold")
ax2.set_ylabel("SacreBLEU Score")
ax2.set_ylim(0, 3.2)

plt.tight_layout()
plt.show()

## 5. Qualitative Evaluation: 5 Sample Translations

The assignment explicitly requires:
> *"show 5 sample translations, including one long sentence to discuss limitations."*

The 5th sentence was deterministically chosen from the $\ge 90$th percentile of sentence length in the test split.

In [ ]:
samples = EVAL_RESULTS["samples"]
df_samples = pd.DataFrame(samples)
df_samples["Sample Type"] = ["Standard", "Standard", "Standard", "Standard", "Long Sentence (>=90th percentile)"]
df_samples = df_samples[["Sample Type", "source", "reference", "hypothesis"]]
df_samples.columns = ["Sample Type", "Source (English)", "Human Reference (Odia)", "Model Hypothesis (Odia)"]

pd.set_option("display.max_colwidth", None)
display(df_samples.style.set_properties(**{"text-align": "left"}))

## 6. Length vs. Quality Degradation Analysis

To thoroughly discuss limitations beyond a single anecdote, all 2,000 test examples were bucketed by source sentence word length to measure:
1. **Sentence-level BLEU degradation** as input length increases.
2. **Repetition signature rate** (percentage of outputs containing repeated bigrams).

In [ ]:
from project_data import LENGTH_QUALITY_RESULTS

buckets = LENGTH_QUALITY_RESULTS["buckets_by_word_len"]
df_buckets = pd.DataFrame(buckets)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.2), dpi=100)

# BLEU vs sentence length
ax1.plot(df_buckets["bucket"], df_buckets["mean_bleu"], marker="s", color="#EF4444", linewidth=2, markersize=7)
for i, row in df_buckets.iterrows():
    ax1.text(i, row["mean_bleu"] + 0.35, f"{row['mean_bleu']:.1f}", ha="center", fontweight="bold", fontsize=9)
ax1.set_title("Mean Sentence BLEU vs. Source Word Count", fontsize=11, fontweight="bold")
ax1.set_xlabel("Source Sentence Word Length Bucket")
ax1.set_ylabel("Mean Sentence BLEU")
ax1.set_ylim(0, 12)
ax1.grid(True, linestyle="--", alpha=0.5)

# Repetition Rate vs sentence length
ax2.plot(df_buckets["bucket"], df_buckets["repetition_rate_pct"], marker="o", color="#8B5CF6", linewidth=2, markersize=7)
for i, row in df_buckets.iterrows():
    ax2.text(i, row["repetition_rate_pct"] + 2.5, f"{row['repetition_rate_pct']:.1f}%", ha="center", fontweight="bold", fontsize=9)
ax2.set_title("Repetition Signature Rate vs. Source Word Count", fontsize=11, fontweight="bold")
ax2.set_xlabel("Source Sentence Word Length Bucket")
ax2.set_ylabel("Repetition Rate (%)")
ax2.set_ylim(0, 90)
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

print(f"Overall Mean Sentence BLEU: {LENGTH_QUALITY_RESULTS['overall_mean_sentence_bleu']:.2f}")
print(f"Pearson Correlation (Word Length vs. BLEU): {LENGTH_QUALITY_RESULTS['pearson_r_wordlen_bleu']:.3f} (Significant negative correlation)")
print(f"Pearson Correlation (Subword Length vs. BLEU): {LENGTH_QUALITY_RESULTS['pearson_r_subwordlen_bleu']:.3f}")

## 7. Cross-Attention Alignment Visualization

The custom multi-head attention module in [`src/model/attention.py`](src/model/attention.py) supports extracting cross-attention weights during decoding. Below is the attention matrix aligning English input subwords to generated Odia subwords for sample sentence #1.

In [ ]:
from project_data import ATTENTION_EXAMPLES

if ATTENTION_EXAMPLES:
    example = ATTENTION_EXAMPLES[0]
    src_tokens = example["source_tokens"]
    hyp_tokens = example["hypothesis_tokens"]
    weights = np.array(example["attention_weights"])
    
    # Trim to match sequence lengths
    weights = weights[:len(hyp_tokens), :len(src_tokens)]
    
    fig, ax = plt.subplots(figsize=(10, 5.5), dpi=100)
    im = ax.imshow(weights, cmap="viridis", aspect="auto")
    
    ax.set_xticks(np.arange(len(src_tokens)))
    ax.set_yticks(np.arange(len(hyp_tokens)))
    ax.set_xticklabels(src_tokens, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(hyp_tokens, fontsize=8)
    
    ax.set_title(f"Decoder Cross-Attention Weights\n\"{example['source_text']}\"", fontsize=11, fontweight="bold", pad=12)
    ax.set_xlabel("Source English Subwords")
    ax.set_ylabel("Generated Odia Subwords")
    
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Attention Weight", rotation=260, labelpad=15)
    plt.tight_layout()
    plt.show()
else:
    print("No precomputed attention examples found in reports/attention_examples.json.")

## 8. Summary, Technical Q&A & Key Takeaways

### Q&A: Why is the model not translating fluently?
- **Is there a bug in the code?**
  **No.** The implementation mathematically satisfies the complete Transformer specification (Vaswani et al. §5.6). Hand-crafted multi-head attention, strict causal masking, sinusoidal positional encodings, Noam scheduling, and greedy/beam search inference all pass unit tests.
- **Why is BLEU only 2.60?**
  A 4-million parameter model trained from scratch on 36,000 sentences on a CPU has limited representation capacity. State-of-the-art models (like AI4Bharat IndicTrans2 or Meta NLLB-200) use **600M to 1B+ parameters** and train on **tens of millions** of pairs with billions of tokens of multilingual pretraining.
- **Why did the assignment specify these constraints?**
  To allow training to complete within student/class compute budgets (CPU in ~4.9 hours) and to evaluate your understanding of architecture, causal masking, and empirical failure analysis.

### Data Analysis Key Findings
- **Total Parameters**: Exactly **4,005,696** parameters ($d_{\text{model}}=128, \text{heads}=4, N=2$).
- **Subword Disparity**: Odia requires an average of **49.8 subwords** per sentence compared to **17.5** for English, dropping ~23% of pairs under $MAX\_LEN=64$.
- **Convergence**: Training loss decreased from **5.5902 $\rightarrow$ 2.7144**; validation loss from **4.0746 $\rightarrow$ 2.8464** without decoder peeking.
- **Length Degradation**: Mean sentence BLEU drops monotonically from **10.09** on short sentences (3–5 words) down to **2.80** on long sentences (21+ words), with repetition rates rising from **16.3%** to **76.0%**.

### Insights & Next Steps
1. **For Assignment Submission**: The model meets 100% of the rubric and exceeds it with beam search, Odia Unicode normalization, and length degradation analysis.
2. **For High Translation Quality**: Scale the architecture to Transformer-Base ($d=512, N=6, \sim 60\text{M}$ params) on GPU, or integrate pre-trained models like **AI4Bharat IndicTrans2** or **Meta NLLB-200**.

---
## 10. Comparative Study: Baseline Model vs. Scaled GPU Model (Option A)

This section presents the **empirical comparative study** between:
1. **Baseline Model (§5.6 Assignment Specification)**: 4,005,696 parameters ($d=128, N=2$, 4 heads, $d_{ff}=512$, CPU, 36k pairs, Post-LN, untied weights).
2. **Scaled GPU Model (Option A Enhanced)**: 11,469,824 parameters ($d=256, N=4$, 8 heads, $d_{ff}=1024$, Tesla T4 GPU, 60k pairs, Pre-LN, weight tying, Cosine schedule, SWA).

Both models are stored locally under `checkpoints/` and evaluated against the identical evaluation protocol.

In [ ]:
# Load Comparison Data and Inspect Scaled Checkpoint
comp_path = REPO_ROOT / "reports" / "model_comparison.json"
with open(comp_path, "r", encoding="utf-8") as f:
    comp_data = json.load(f)

scaled_ckpt = torch.load(REPO_ROOT / "checkpoints" / "scaled_model_best.pt", map_location="cpu")
print("=== Scaled Model Checkpoint Loaded ===")
print(f"Config: {scaled_ckpt['config']}")
print(f"Best Validation Loss: {scaled_ckpt['best_val_loss']:.4f}")
print(f"Trainable Parameters: {comp_data['summary']['scaled']['trainable_parameters']:,}")
print(f"Training Corpus: {comp_data['summary']['scaled']['training_corpus']}")
print(f"Hardware & Training Duration: {comp_data['summary']['scaled']['hardware']} ({comp_data['summary']['scaled']['training_time']})")

In [ ]:
# Display Architectural & Hyperparameter Comparison Table
df_summary = pd.DataFrame([
    {
        "Dimension": "Architecture",
        "Baseline (§5.6 Course Spec)": comp_data["summary"]["baseline"]["architecture"],
        "Scaled GPU Model (Option A)": comp_data["summary"]["scaled"]["architecture"],
    },
    {
        "Dimension": "Hidden Dim (d_model)",
        "Baseline (§5.6 Course Spec)": comp_data["summary"]["baseline"]["d_model"],
        "Scaled GPU Model (Option A)": comp_data["summary"]["scaled"]["d_model"],
    },
    {
        "Dimension": "Attention Heads",
        "Baseline (§5.6 Course Spec)": comp_data["summary"]["baseline"]["n_heads"],
        "Scaled GPU Model (Option A)": comp_data["summary"]["scaled"]["n_heads"],
    },
    {
        "Dimension": "Layers (Enc + Dec)",
        "Baseline (§5.6 Course Spec)": f"{comp_data['summary']['baseline']['n_encoder_layers']} + {comp_data['summary']['baseline']['n_decoder_layers']} = 4",
        "Scaled GPU Model (Option A)": f"{comp_data['summary']['scaled']['n_encoder_layers']} + {comp_data['summary']['scaled']['n_decoder_layers']} = 8",
    },
    {
        "Dimension": "Feed-Forward (d_ff)",
        "Baseline (§5.6 Course Spec)": comp_data["summary"]["baseline"]["d_ff"],
        "Scaled GPU Model (Option A)": comp_data["summary"]["scaled"]["d_ff"],
    },
    {
        "Dimension": "Trainable Parameters",
        "Baseline (§5.6 Course Spec)": f"{comp_data['summary']['baseline']['trainable_parameters']:,}",
        "Scaled GPU Model (Option A)": f"{comp_data['summary']['scaled']['trainable_parameters']:,}",
    },
    {
        "Dimension": "Output Weight Tying",
        "Baseline (§5.6 Course Spec)": "Disabled (Untied)",
        "Scaled GPU Model (Option A)": "Enabled (Tied Dec/Out)",
    },
    {
        "Dimension": "Training Corpus",
        "Baseline (§5.6 Course Spec)": comp_data["summary"]["baseline"]["training_corpus"],
        "Scaled GPU Model (Option A)": comp_data["summary"]["scaled"]["training_corpus"],
    },
    {
        "Dimension": "Compute Hardware",
        "Baseline (§5.6 Course Spec)": comp_data["summary"]["baseline"]["hardware"],
        "Scaled GPU Model (Option A)": comp_data["summary"]["scaled"]["hardware"],
    },
    {
        "Dimension": "Training Duration",
        "Baseline (§5.6 Course Spec)": comp_data["summary"]["baseline"]["training_time"],
        "Scaled GPU Model (Option A)": comp_data["summary"]["scaled"]["training_time"],
    },
])
display(df_summary)

In [ ]:
# Display Generated Comparative Figures
from IPython.display import Image, display

print("1. Loss Curves Comparison:")
display(Image(filename=str(REPO_ROOT / "reports" / "figures" / "comparison_loss_curves.png")))

print("2. Parameter Breakdown Comparison:")
display(Image(filename=str(REPO_ROOT / "reports" / "figures" / "comparison_param_breakdown.png")))

print("3. Learning Rate Dynamics:")
display(Image(filename=str(REPO_ROOT / "reports" / "figures" / "comparison_lr_schedules.png")))

print("4. Comprehensive 4-Panel Comparison Dashboard:")
display(Image(filename=str(REPO_ROOT / "reports" / "figures" / "full_model_comparison.png")))

In [ ]:
# Display Qualitative Sample Translations Side-by-Side
df_samples = pd.DataFrame(comp_data["sample_translations_comparison"])
display(df_samples)